# Generate Expected Outputs

**What this notebook does, in plain words:**

Our test case files (in `test_cases/<Topic>/*.json`) have inputs but every
`expected_output` is still `null`. We never ran the real solution against
those inputs — we only wrote the inputs down.

This notebook fixes that:

1. For every problem, take the real editorial solution (`dsa_data.json`) and glue it
   together with that problem's `wrapper_main` (`starter_code.json`) — this gives us one
   complete, runnable `.cpp` program.
2. For every test case of that problem, send that program + the test case's input
   to **CodeForge RCE** (our own Docker-based code runner) and ask it to run it.
3. Take whatever the program printed (`stdout`) and save it as `expected_output`.
4. Write the file back to disk, so next time we load it, the outputs are already there.

It is **resumable** — if you run it twice, or it dies halfway through, test cases that
already have a non-null `expected_output` are skipped, so you never overwrite good data
or waste time re-running things that already worked.


## Step 1 — Imports

Nothing fancy: `json` to read/write our files, `pathlib.Path` to walk the
`test_cases` folder, and `os`/`time` for small housekeeping.


In [32]:
import json
import os
import time
from pathlib import Path


## Step 2 — Connect to CodeForge RCE

`CodeForgeRCE` is the engine that actually compiles and runs C++ code inside a
throwaway Docker container, for us. `ExecutionRequest` is just a small data object
that describes *one* thing we want run: the code, the language, and stdin to feed it.

This needs Docker running on your machine (Docker Desktop, or `dockerd`), because
under the hood CodeForge spins up a container per run.

If this import fails with `ModuleNotFoundError`, install it first, e.g.:
```
pip install -e /path/to/your/local/CodeForge-RCE
```
(or `pip install git+https://github.com/DevXDividends/CodeForge-RCE.git`)

**One setting we override:** CodeForge rejects any stdin longer than
`MAX_STDIN_CHARS` (1,000,000 characters by default) as a safety cap. Some of our
own stress-test cases (e.g. `n = 1,000,000` array elements) produce a raw input
string several times bigger than that, purely because that many numbers take that
many characters to write out — not because anything is wrong with the test case.
`MAX_STDIN_CHARS` is a plain attribute on the `rce` object, so we can raise it for
our own use without touching CodeForge's source at all.

**The same thing happens on the way OUT.** CodeForge's executor stops a program after
it prints 1 MiB (`max_output_bytes`) and reports `OUTPUT_LIMIT_EXCEEDED`. A linked-list
problem whose input has 1,000,000 nodes legitimately prints ~7 MB of answer, so we raise
this cap too. The executor object is created lazily (on the first run), so we ask the
engine to build its services first, then change the number on `rce.executor`.


In [33]:
from codeforge import CodeForgeRCE, ExecutionRequest

# One shared engine instance — it's fine to reuse this for every single run,
# CodeForge gives each execute() call its own isolated workspace + container.
rce = CodeForgeRCE()

# Raise the stdin size cap so our large stress-test cases (n up to ~1,000,000
# elements) aren't rejected before they even reach Docker. 20,000,000 chars is
# comfortably above anything our test generator produces, with headroom to spare.
rce.MAX_STDIN_CHARS = 20_000_000


def apply_limits():
    """Raise the output cap on CodeForge's executor (default is only 1 MiB).
    Safe to call many times; cheap once the services are already built."""
    rce._initialize_services()  # builds rce.executor if Docker is up
    if rce.executor is not None:
        rce.executor.max_output_bytes = 64 * 1024 * 1024  # 64 MiB


apply_limits()
print("CodeForge engine ready 👍")


CodeForge engine ready 👍


## Step 3 — Load our data files

- `dsa_data.json` → the **real editorial solutions** (the actual working code),
  grouped by topic: `dsa_data[topic][problem_key] = code`
- `starter_code.json` → each problem's `wrapper_main` (the `#include`s + `main()`
  that reads input, calls the solution, and prints the answer)


In [34]:
with open("../dsa_data.json", "r", encoding="utf-8") as file:
    dsa_data = json.load(file)

with open("../starter_code.json", "r", encoding="utf-8") as file:
    starter_code = json.load(file)

print("Loaded dsa_data.json and starter_code.json 👍")


Loaded dsa_data.json and starter_code.json 👍


## Step 4 — Flatten the editorials into one simple lookup

`dsa_data.json` is nested by topic (`dsa_data["Stack"]["Min Stack.cpp"]`), but our
test case files only ever give us the `problem_key` (e.g. `"Min Stack.cpp"`), not the
topic. So instead of hunting through every topic each time, we flatten it once into a
single dictionary: `editorial_code["Min Stack.cpp"] = "...the real code..."`.


In [35]:
editorial_code = {}

for topic in dsa_data:
    for problem_key in dsa_data[topic]:
        editorial_code[problem_key] = dsa_data[topic][problem_key]

print(f"Flattened {len(editorial_code)} editorial solutions from {len(dsa_data)} topics")


Flattened 178 editorial solutions from 10 topics


## Step 5 — Build the full, runnable program for one problem

This turned out trickier than "just glue two strings together". Three real problems
showed up once we actually tried compiling every problem (178 of them, with real g++):

**Problem A — include order.** The editorial's `class Solution` uses types like
`vector`/`string`, but only `wrapper_main` carries `#include <bits/stdc++.h>`. If we
paste `editorial_code` BEFORE `wrapper_main`, the compiler reaches `vector` before it
has ever seen the include — error.

**Problem B — multiple approaches in one editorial.** ~35% of editorials store more
than one approach in the same string (brute force AND optimal), each as its own
`class Solution { ... }` (or `class MinStack { ... }`, etc). Pasting the whole string
gives the compiler the same class twice — "redefinition" error. We keep only the
**last** block for each class/struct name (the final/optimal approach).

**Problem C — helper types belong to wrapper_main, not the editorial.** For Linked
List / Tree problems, `wrapper_main` defines its own `Node`/`TreeNode`/`ListNode`
struct (so it can build the test's input) — and that struct must exist BEFORE the
editorial's `Solution` class, which uses it as a parameter type. So `wrapper_main`
can't be treated as one opaque blob pasted after the editorial — we need to split it
at `int main(` into a **preamble** (includes + helper types/functions) and the
**main() itself**, and sandwich the editorial in between:

```
[ wrapper_main's preamble ]   <- #include's + any Node/TreeNode struct it defines
[ editorial's class(es)   ]   <- the real solution, last approach only
[ wrapper_main's int main ]   <- reads stdin, calls the solution, prints the answer
```

If the editorial also needs a *second* class besides `Solution` (like `BSTIterator`
for one approach, or `Node` for a linked list), we keep that too — but ordered by
where it was **last used** in the editorial text, so a helper class a later approach
depends on still lands before that approach, not after it.


In [36]:
import re


def strip_block_comments(code):
    """Removes /* ... */ comments. A few editorials keep an old class definition
    inside a comment purely as documentation -- we don't want our parser to
    mistake that for a real, compilable class."""
    return re.sub(r"/\*.*?\*/", "", code, flags=re.DOTALL)


### A small helper: find every `class X { ... }` / `struct X { ... }` block

We can't just regex-match up to the first `}` — a class body usually contains many
`{`/`}` pairs of its own (if-blocks, loops, nested lambdas...). So we find where the
class's own `{` starts, then walk forward counting `{` as +1 and `}` as -1; when the
count returns to 0, that's the class's real closing brace. This technique is called
**brace matching / brace counting** — a simple, standard way to find a balanced block
without writing a full C++ parser.


In [37]:
TYPE_KEYWORD = re.compile(r"\b(class|struct)\s+(\w+)")


def extract_type_blocks(code):
    """Returns a list of (name, full_text) for every top-level class/struct block
    found in `code`, in the order they appear."""

    blocks = []
    i, n = 0, len(code)

    while True:
        match = TYPE_KEYWORD.search(code, i)
        if match is None:
            break

        name = match.group(2)
        brace_start = code.find("{", match.end())
        if brace_start == -1:
            i = match.end()
            continue

        # Walk forward counting braces until we're back to depth 0 -- that's
        # this class's real closing brace, no matter how much is nested inside.
        depth = 0
        j = brace_start
        while j < n:
            if code[j] == "{":
                depth += 1
            elif code[j] == "}":
                depth -= 1
                if depth == 0:
                    break
            j += 1

        end = j + 1

        # swallow the trailing ';' that C++ requires after a class/struct definition
        k = end
        while k < n and code[k] in " \t\n":
            k += 1
        if k < n and code[k] == ";":
            end = k + 1

        blocks.append((name, code[match.start():end]))
        i = end

    return blocks


### Now the real `get_full_code()`


In [38]:
def get_full_code(problem_key):
    """Returns one complete, compilable C++ program for this problem, or raises
    a clear error if a piece is missing."""

    if problem_key not in editorial_code:
        raise KeyError(f"No editorial solution found for '{problem_key}' in dsa_data.json")

    if problem_key not in starter_code:
        raise KeyError(f"No starter code found for '{problem_key}' in starter_code.json")

    # Split wrapper_main into: everything before int main() (includes + any
    # helper struct/functions it defines), and int main() itself.
    wrapper_main = starter_code[problem_key]["wrapper_main"]
    main_index = wrapper_main.find("int main(")
    preamble = wrapper_main[:main_index]
    main_part = wrapper_main[main_index:]

    # Any class/struct name wrapper_main's preamble already defines (e.g. a
    # Node/TreeNode struct built for this test's input) -- the editorial must
    # not redefine these.
    preamble_names = {name for name, _ in extract_type_blocks(preamble)}

    # Clean the editorial (drop commented-out leftovers) and pull out every
    # class/struct block it defines.
    solution = strip_block_comments(editorial_code[problem_key])
    editorial_types = extract_type_blocks(solution)

    # For each name, remember its LAST block (collapses multiple approaches
    # down to the final one) and the position of its LAST occurrence (so a
    # helper class a later approach depends on still gets ordered correctly).
    last_block_by_name = {}
    last_pos_by_name = {}
    for position, (name, block) in enumerate(editorial_types):
        last_block_by_name[name] = block
        last_pos_by_name[name] = position

    ordered_names = sorted(last_pos_by_name, key=lambda name: last_pos_by_name[name])

    kept_blocks = [
        last_block_by_name[name]
        for name in ordered_names
        if name not in preamble_names  # wrapper_main already provides this type
    ]

    editorial_final = "\n\n".join(kept_blocks)

    return preamble + "\n" + editorial_final + "\n" + main_part


## Step 6 — Run one test case through CodeForge

This is the actual "generate the output" step. We hand CodeForge the full program
and the test case's input (as stdin), and it compiles + runs it inside Docker for us.

`rce.execute()` always returns a plain dict — it never raises. If everything went
fine, `result["status"] == "SUCCESS"` and `result["stdout"]` holds the program's output.
Anything else (`COMPILATION_ERROR`, `TLE`, `RUNTIME_ERROR`, ...) means something is
wrong with that problem's code or test case, and we should NOT invent an expected
output — we flag it instead.


In [39]:
def run_one(code, stdin_text):
    """Runs `code` with `stdin_text` as input via CodeForge.
    Returns (output_text, None) on success, or (None, error_message) on failure."""

    request = ExecutionRequest(
        code=code,
        language="cpp",
        stdin=stdin_text,
        timeout=5,  # seconds — generous, since we're just generating outputs, not judging speed
    )

    apply_limits()  # make sure the raised output cap is in place
    result = rce.execute(request)

    if result["status"] == "SUCCESS":
        return result["stdout"].strip(), None

    # Something went wrong before/while running. 'logs' exists for compile-time
    # failures, 'stdout' exists for runtime failures (TLE, crash, etc).
    reason = result.get("logs") or result.get("stdout") or "(no details)"
    error_message = f"{result['status']}: {reason}"
    return None, error_message


## Step 7 — Process one whole test-case file

For a single `<Problem>_test_cases.json` file:

1. Load it, read its `problem_key`.
2. Build that problem's full program **once** (not once per test case — it's the same
   program every time, only the stdin changes).
3. For each test case:
   - if `expected_output` is already filled in → **skip it** (this is what makes the
     notebook resumable / safe to re-run).
   - otherwise → run it through CodeForge and fill it in.
4. Save the file back to disk — but safely: write to a temporary file first, then swap
   it into place. This way, if the notebook crashes mid-write, the original file is
   never left half-written or corrupted.


In [40]:
def process_file(file_path):
    """Fills in expected_output for every test case in one file. Returns a small
    summary dict so we can print a report at the end."""

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    problem_key = data["problem_key"]

    summary = {"file": file_path.name, "generated": 0, "skipped": 0, "failed": 0}

    # If we can't even build the program for this problem, there's no point
    # looping over its test cases — flag it once and move on.
    try:
        full_code = get_full_code(problem_key)
    except KeyError as e:
        print(f"⚠️  Skipping whole file '{file_path.name}': {e}")
        summary["failed"] = len(data["test_cases"])
        return summary

    changed = False

    for test_case in data["test_cases"]:

        # Resumability: a test case that already has an output is left untouched.
        if test_case["expected_output"] is not None:
            summary["skipped"] += 1
            continue

        stdin_text = test_case["input"]["raw"]
        output, error = run_one(full_code, stdin_text)

        if error is not None:
            print(f"❌ {problem_key} — a test case failed: {error}")
            summary["failed"] += 1
            continue

        test_case["expected_output"] = output
        summary["generated"] += 1
        changed = True

    # Only touch the file on disk if we actually generated something new.
    if changed:
        temp_path = file_path.with_suffix(".tmp")
        with open(temp_path, "w", encoding="utf-8") as file:
            json.dump(data, file, indent=2, ensure_ascii=False)
        os.replace(temp_path, file_path)  # atomic swap — no half-written file ever

    return summary


## Step 8 — Run it across every test case file

`Path("../test_cases").rglob("*.json")` walks every topic folder and finds every
`*_test_cases.json` file, regardless of which subfolder it's in.

We process them one at a time (not in parallel) — simple, and easy to reason about
while we're still learning how this pipeline behaves. Parallelizing this later is a
small, optional improvement once we trust it works.


In [41]:
# Some problems have 2 test files with identical inputs (an old one and a *_FIXED copy).
# Running both would just double the work, so we run the first one per problem and
# warn about the rest. Originals (without _FIXED) are preferred.
all_files = sorted(Path("../test_cases").rglob("*.json"), key=lambda p: ("_FIXED" in p.name, str(p)))

test_case_files = []
seen_problem_keys = set()
for file_path in all_files:
    with open(file_path, "r", encoding="utf-8") as f:
        key = json.load(f)["problem_key"]
    if key in seen_problem_keys:
        print(f"⚠️  Duplicate test file for '{key}', skipping: {file_path.name}  (consider deleting it)")
        continue
    seen_problem_keys.add(key)
    test_case_files.append(file_path)

print(f"Found {len(test_case_files)} test case files\n")

all_summaries = []
start_time = time.time()

for file_path in test_case_files:
    summary = process_file(file_path)
    all_summaries.append(summary)
    print(f"✅ {summary['file']:60s} generated={summary['generated']:3d}  skipped={summary['skipped']:3d}  failed={summary['failed']:3d}")

elapsed = time.time() - start_time

total_generated = sum(s["generated"] for s in all_summaries)
total_skipped = sum(s["skipped"] for s in all_summaries)
total_failed = sum(s["failed"] for s in all_summaries)

print("\n--------------------------------")
print(f"Files processed : {len(all_summaries)}")
print(f"Outputs generated : {total_generated}")
print(f"Already had output (skipped) : {total_skipped}")
print(f"Failed : {total_failed}")
print(f"Time taken : {elapsed:.1f}s")
print("--------------------------------")


Found 165 test case files

✅ CSES - Apartments_test_cases.json                            generated=  0  skipped= 20  failed=  0
✅ Count Number of Pairs With Absolute Difference K_test_cases.json generated=  0  skipped= 20  failed=  0
✅ Design HashMap_test_cases.json                               generated=  0  skipped= 20  failed=  0
✅ Design HashSet_test_cases.json                               generated=  0  skipped= 20  failed=  0
✅ Find Pair Given Difference_test_cases.json                   generated=  0  skipped= 20  failed=  0
✅ Merge-Sorted-Array_test_cases.json                           generated=  0  skipped= 20  failed=  0
✅ Number of Subsequences That Satisfy the Given Sum Condition_test_cases.json generated=  0  skipped= 20  failed=  0
✅ Palindrome number test cases .json                           generated=  0  skipped= 20  failed=  0
✅ Plus-One_test_cases.json                                     generated=  0  skipped= 20  failed=  0
✅ Remove-Duplicates-from-Sorted-Arra

## Step 9 (optional) — Sanity check: look at any problems that failed

Run this if `total_failed > 0` above, to see exactly which files had problems.
Common causes: a problem's `wrapper_main` input format doesn't match the test case's
`raw` input, or the editorial code itself has a bug.


In [42]:
failed_files = [s["file"] for s in all_summaries if s["failed"] > 0]

if failed_files:
    print("Files with at least one failure:")
    for name in failed_files:
        print(f"  - {name}")
else:
    print("No failures 🎉")


Files with at least one failure:
  - Remove every k'th node_test_cases.json


## Step 10 (optional) — Try a single problem by hand

Useful while learning / debugging: pick one problem key and watch exactly what
CodeForge does with it, without touching any files on disk.


In [43]:
# Example — change this to any key that exists in starter_code.json
example_key = "Aggressive-cows.cpp"

example_code = get_full_code(example_key)
print(example_code)
print("\n--- running with a sample input ---\n")

output, error = run_one(example_code, "5\n1 2 8 4 9\n3")
print("Output:", output)
print("Error:", error)


#include <bits/stdc++.h>
using namespace std;


class Solution {
  public:
  bool check(vector<int> &a,int dist,int cows){
      int cnt=1;
      int prev=a[0];
      for(int i=1;i<a.size();i++){
          if(a[i]-prev>=dist){
              cnt++;
              prev=a[i];
          }
      }
      return (cnt>=cows);
  }
    int aggressiveCows(vector<int> &stalls, int k) {
        // code here
        sort(stalls.begin(),stalls.end());
        int res=-1;
        
        int low=0,high=stalls.back()-stalls[0];
        while(low<=high){
            int mid=low+(high-low)/2;
            if(check(stalls,mid,k)){
                res=mid;
                low=mid+1;
            }else high=mid-1;
        }
        return res;
    }
};
int main() {
    int stalls_n; cin >> stalls_n;
    vector<int> stalls(stalls_n);
    for (int i = 0; i < stalls_n; i++) cin >> stalls[i];
    int k;
    cin >> k;
    
    Solution obj;
    auto ans = obj.aggressiveCows(stalls, k);
    cout << ans;
}

--- runn